# Music, Brain & Wellbeing: Feature Engineering & Preprocessing Foundation

This notebook implements the preprocessing pipeline, train/test split, scaling, categorical encoding, and leakage prevention systems for our model-ready dataset.

We build this from first principles:
1. Define the input features ($X$) and target ($y$) explicitly.
2. Separate variables into distinct feature groups.
3. Split our dataset into Train and Test sets before any transformation to avoid data leakage.
4. Construct a robust preprocessing pipeline using scikit-learn's `ColumnTransformer` and `Pipeline`.
5. Implement a reusable preprocessing module.



## 1. Define Features ($X$) and Target ($y$)

We load our cleaned dataset `data/processed/mxmh_cleaned.csv`. 

### Defining $X$ and $y$:
* **$X$ (Features)**: The input information used by the machine learning model to learn and make predictions.
* **$y$ (Target)**: The answer or outcome we want our model to learn to predict.

### Preventing Target Leakage:
We choose **`Anxiety`** as our target ($y$).
To prevent target leakage, we must exclude `Depression`, `Insomnia`, and `OCD` from our features ($X$). These are concurrent self-reported psychological symptom scores collected in the same survey. Including them would make predicting `Anxiety` trivial and artificially boost performance, but would not serve our goal of predicting wellbeing outcomes from music listening habits alone.



In [1]:
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv("data/processed/mxmh_cleaned.csv")

# y is the target we want to predict (Anxiety)
y = df["Anxiety"]

# X represents our feature space
# We exclude the target Anxiety and secondary target columns to prevent target leakage
leakage_cols = ["Anxiety", "Depression", "Insomnia", "OCD"]
X = df.drop(columns=leakage_cols)

print("Target variable shape (y):", y.shape)
print("Feature space shape (X):", X.shape)
print("\nFirst 3 rows of target (y):")
print(y.head(3))
print("\nFirst 3 rows of features (X):")
print(X.head(3))



Target variable shape (y): (736,)
Feature space shape (X): (736, 27)

First 3 rows of target (y):
0    3.0
1    7.0
2    7.0
Name: Anxiety, dtype: float64

First 3 rows of features (X):
    Age Primary streaming service  Hours per day While working  \
0  18.0                   Spotify            3.0           Yes   
1  63.0                   Pandora            1.5           Yes   
2  18.0                   Spotify            4.0            No   

  Instrumentalist Composer         Fav genre Exploratory Foreign languages  \
0             Yes      Yes             Latin         Yes               Yes   
1              No       No              Rock         Yes                No   
2              No       No  Video game music          No               Yes   

     BPM  ...  Frequency [K pop]  Frequency [Latin]  Frequency [Lofi]  \
0  156.0  ...                  3                  3                 1   
1  119.0  ...                  1                  2                 1   
2  132.0  ...    

## 2. Review Features and Dtypes

We group our features into `Numerical` and `Categorical` groups based on their Pandas data types (`dtypes`).

* **Numerical Features**: Features represented by continuous or integer numeric values.
* **Categorical Features**: Features represented by discrete textual classes.



In [2]:
# Inspect dtypes of our feature matrix X
print("=== Pandas Feature Types ===")
print(X.dtypes)

# Group features based on their dtypes
categorical_cols = list(X.select_dtypes(include=["object", "category"]).columns)
numerical_cols = list(X.select_dtypes(include=["int64", "float64"]).columns)

print("\nCategorical columns found:", len(categorical_cols))
print(categorical_cols)
print("\nNumerical columns found:", len(numerical_cols))
print(numerical_cols)



=== Pandas Feature Types ===
Age                             float64
Primary streaming service           str
Hours per day                   float64
While working                       str
Instrumentalist                     str
Composer                            str
Fav genre                           str
Exploratory                         str
Foreign languages                   str
BPM                             float64
Frequency [Classical]             int64
Frequency [Country]               int64
Frequency [EDM]                   int64
Frequency [Folk]                  int64
Frequency [Gospel]                int64
Frequency [Hip hop]               int64
Frequency [Jazz]                  int64
Frequency [K pop]                 int64
Frequency [Latin]                 int64
Frequency [Lofi]                  int64
Frequency [Metal]                 int64
Frequency [Pop]                   int64
Frequency [R&B]                   int64
Frequency [Rap]                   int64
Frequency [

C:\Users\aksha\AppData\Local\Temp\ipykernel_34480\2788590339.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = list(X.select_dtypes(include=["object", "category"]).columns)


## 3. Train / Test Split

### Why do we split data?
To evaluate how well our model generalizes to new, unseen observations. If we train and evaluate our model on the same data, the model can simply memorize the training data (overfit), giving us a false impression of high accuracy.

### Key concepts:
* **Training set**: The subset of data used by the model to learn relationships and fit parameters.
* **Test set**: The independent subset of data kept hidden from the model during training, used strictly for final evaluation.

We perform a random split with a fixed `random_state` to ensure reproducibility.



In [3]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)



Training features shape: (588, 27)
Testing features shape: (148, 27)
Training target shape: (588,)
Testing target shape: (148,)


## 4. Preprocessing and ColumnTransformer

Different types of features require different preprocessing steps:

### A. Numerical Preprocessing:
1. **Imputation**: Although our cleaned dataset does not have missing values, production pipelines need to handle missing inputs robustly. We use `SimpleImputer` with `strategy='median'`.
2. **Scaling**: Distance-based and linear models (like Linear Regression, Logistic Regression, KNN, and SVM) are sensitive to feature scales. If one feature ranges from 10 to 1000 and another ranges from 0 to 1, the model will treat the larger feature as more important. We use `StandardScaler` to scale numerical features to have a mean of 0 and variance of 1.
* *Note*: Tree-based models (like Decision Trees, Random Forests, and Gradient Boosting) are scale-invariant and do not require scaling.

### B. Categorical Preprocessing:
1. **Imputation**: We handle potential missing classes using `SimpleImputer(strategy='most_frequent')`.
2. **Encoding**: Machine learning algorithms require numeric inputs. We use `OneHotEncoder(handle_unknown='ignore')` to convert nominal categorical strings into binary columns.



In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define Numerical Preprocessing Pipeline
# SimpleImputer replaces NaN with median, StandardScaler standardizes to mean=0, std=1
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Define Categorical Preprocessing Pipeline
# SimpleImputer replaces NaN with mode, OneHotEncoder converts classes to binary columns
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine pipelines using ColumnTransformer
# ColumnTransformer applies specific pipelines to selected columns
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])

print("ColumnTransformer constructed successfully.")



ColumnTransformer constructed successfully.


## 5. Preventing Data Leakage

### What is Data Leakage?
Data leakage occurs when information from outside the training dataset is used to train the model. This typically happens when preprocessing statistics (like the mean/variance for scaling, or modes for imputation) are calculated using the *entire* dataset before splitting. If we compute the global mean, information from our test set "leaks" into the training set, producing overly optimistic test results that fail in production.

### How to avoid leakage:
We must **ONLY** call `.fit()` or `.fit_transform()` on our **Training set** (`X_train`). This learns the scaling means, variances, and categorical categories exclusively from training data.
We then call `.transform()` (without fitting) on our **Test set** (`X_test`) to apply those learned values.



In [5]:
# Fit preprocessor strictly on X_train
preprocessor.fit(X_train)

# Transform training and testing feature sets
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Fit and transform completed safely without data leakage.")
print("Processed training features shape:", X_train_processed.shape)
print("Processed testing features shape:", X_test_processed.shape)



Fit and transform completed safely without data leakage.
Processed training features shape: (588, 54)
Processed testing features shape: (148, 54)


## 6. Pipeline Verification and Feature Names

We inspect the resulting shapes and verify the column names generated after one-hot encoding.



In [6]:
# Get generated feature names from the OneHotEncoder step
ohe_feature_names = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_cols)

# Combine original numerical features with new one-hot features
final_feature_names = list(numerical_cols) + list(ohe_feature_names)

print(f"Total features input: {X.shape[1]}")
print(f"Total features after encoding: {len(final_feature_names)}")
print("First 10 final feature names:")
print(final_feature_names[:10])

# Verify there are no NaN values remaining in processed matrices
print("\nMissing values in processed training set:", np.isnan(X_train_processed).sum())
print("Missing values in processed testing set:", np.isnan(X_test_processed).sum())



Total features input: 27
Total features after encoding: 54
First 10 final feature names:
['Age', 'Hours per day', 'BPM', 'Frequency [Classical]', 'Frequency [Country]', 'Frequency [EDM]', 'Frequency [Folk]', 'Frequency [Gospel]', 'Frequency [Hip hop]', 'Frequency [Jazz]']

Missing values in processed training set: 0
Missing values in processed testing set: 0


## 7. Pipeline with Model Placeholder

We package the preprocessing steps and a model placeholder (a simple dummy regressor that predicts the mean of the training target) into a single scikit-learn `Pipeline`.
This verifies our preprocessing pipeline executes seamlessly when paired with a predictor.



In [7]:
from sklearn.dummy import DummyRegressor

# Create baseline placeholder pipeline
# DummyRegressor predicts target mean; acts as a baseline to verify integration
baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", DummyRegressor(strategy="mean"))
])

# Fit baseline pipeline on training data
baseline_pipeline.fit(X_train, y_train)

# Generate baseline predictions to verify the pipeline works
y_pred_dummy = baseline_pipeline.predict(X_test)
print("Baseline placeholder pipeline predictions (first 5):")
print(y_pred_dummy[:5])



Baseline placeholder pipeline predictions (first 5):
[5.82653061 5.82653061 5.82653061 5.82653061 5.82653061]


## 8. Included and Excluded Features

Below we document our feature selection choices:

### Excluded Features:
* **`Timestamp`**: Excluded because it is survey metadata (submission time) and does not represent a physical or behavioral attribute.
* **`Permissions`**: Excluded because it has zero variance (all values are "I understand.") and provides no information.
* **`Depression`, `Insomnia`, `OCD`**: Excluded because they are concurrent mental health target indicators. Including them would introduce target leakage if our goal is to predict anxiety from music listening habits.

### Included Features:
* **Demographics**: `Age`
* **Music Behaviour**: `Hours per day`, `While working`, `Instrumentalist`, `Composer`, `Exploratory`, `Foreign languages`, `Primary streaming service`
* **Music Preferences**: `Fav genre`, `BPM`
* **Listening Context**: 16 ordinal `Frequency [genre]` variables
* **Other relevant variables**: `Music effects`



---
## Interview Explanation

Here we answer standard Data Science technical interview questions related to preprocessing and pipelines:

### 1. Why can't we feed raw categorical strings directly into most classical ML algorithms?
Most classical machine learning algorithms (like Linear Regression, SVM, and Neural Networks) are based on mathematical calculations such as matrix multiplications and distance metrics. These operations require numeric vectors. Raw text strings like "Spotify" or "Rock" cannot be mathematically multiplied or differentiated. Thus, they must be mapped to numeric values first.

### 2. Why do we split train and test data?
The goal of machine learning is to predict accurately on *new, unseen data*. If we train and evaluate a model on the same dataset, it can memorize the inputs (overfit), which inflates training performance. Splitting allows us to train on one set (training set) and use an independent set (test set) to simulate real-world generalization and obtain an unbiased estimate of model performance.

### 3. Why is preprocessing inside a Pipeline?
Putting preprocessing inside a `Pipeline` ensures that scaling, imputation, and encoding parameters (like training mean, standard deviation, and modes) are fitted *only* on the training set. When we call `.predict()` on new test data, the pipeline automatically applies those pre-fit training parameters. This prevents the model from seeing future test information, completely eliminating data leakage.

### 4. What is data leakage?
Data leakage occurs when information from outside the training dataset is inadvertently used to train a model. Common examples include scaling the entire dataset before splitting (leaking the test mean), or including target-correlated fields (like concurrent diagnosis indicators) that wouldn't be available at inference time. This leads to overly optimistic evaluation metrics during testing but poor performance when deployed.

### 5. Why might scaling matter for Logistic Regression?
Logistic Regression calculates coefficients/weights for each feature. If features have very different scales (e.g. Age ranging from 10 to 89 vs Daily hours ranging from 0 to 24), the model coefficients will vary wildly in magnitude to compensate. Furthermore, if L1/L2 regularization is applied, features with larger scales will be penalized disproportionately. Scaling ensures equal regularization penalties and faster gradient descent convergence.

### 6. Why is scaling usually less important for Decision Trees?
Decision Trees make splits by evaluating individual features one at a time (e.g. `Age > 30`). They do not sum feature weights or compute distance metrics across multiple features. Since the split criteria are scale-invariant, multiplying a feature by a constant has no effect on the tree's split selections.

### 7. Why use OneHotEncoder?
`OneHotEncoder` converts nominal (non-ordered) categorical variables into binary columns. This prevents the model from assuming an artificial mathematical rank. For instance, if we label encoded Spotify as 1, Apple Music as 2, and Pandora as 3, a linear model would assume Pandora is "greater" than Spotify and Apple Music, which is mathematically invalid.

